# VSLink — Ghép câu: Mức 2 (fine-tune)

Học trực tiếp *"dãy top-3 có nhiễu → câu đúng"* trên dữ liệu tự sinh (khung câu + quy tắc NNKH +
nhiễu nhận dạng). Ý tưởng: Gloss2Text (EMNLP Findings 2024, LoRA), HyPoradise (NeurIPS 2023, sửa lỗi
từ N-best), Moryossef và cs. (2021, sinh dữ liệu gloss bằng quy tắc).

- **A. ViT5-base** (VietAI): seq2seq tiếng Việt, train toàn bộ, nhanh.
- **B. Qwen-2.5-Instruct + QLoRA** (Unsloth): đổi `MO_HINH_QWEN` để so 1.5B / 3B / 7B.

GPU: Kaggle T4. Đánh giá trên **tập kiểm tra viết tay** (`du-lieu/tap_kiem_tra.jsonl`), không phải dữ liệu sinh.

In [ ]:
# Lay code (nhanh tinh-nang/ghep-cau). Kaggle: Settings -> Internet ON, Accelerator GPU T4 x2.
!rm -rf vslink && git clone -q --depth 1 -b tinh-nang/ghep-cau https://github.com/VSLinkProject/vslink.git
%cd vslink/nghien-cuu/ghep-cau
!pip install -q sacrebleu

In [ ]:
# (Nen co) Nhieu nhan dang THAT: tai du_doan_test.jsonl (xuat bang xuat_top5_kaggle.py tu notebook
# huan luyen) len Kaggle thanh mot Dataset, "Add Input", roi sua duong dan duoi day.
import glob, shutil
tim = glob.glob('/kaggle/input/**/du_doan_test.jsonl', recursive=True)
if tim:
    shutil.copy(tim[0], 'du-lieu/du_doan_test.jsonl'); print('Dung nhieu THAT:', tim[0])
else:
    print('Chua co du_doan_test.jsonl -> dung nhieu GIA LAP (tam thoi, khong dung cho bao cao cuoi)')
!python tao_du_lieu.py

## A. ViT5-base

In [ ]:
import json, torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq,
                          Seq2SeqTrainer, Seq2SeqTrainingArguments)

def doc(tep):
    return [json.loads(d) for d in open(tep, encoding='utf-8')]
tr, va = doc('du-lieu/sinh/train.jsonl'), doc('du-lieu/sinh/val.jsonl')

T5_GOC = 'VietAI/vit5-base'
tk = AutoTokenizer.from_pretrained(T5_GOC)
m = AutoModelForSeq2SeqLM.from_pretrained(T5_GOC)

def ma_hoa(b):
    x = tk(b['dau_vao'], max_length=256, truncation=True)
    x['labels'] = tk(text_target=b['cau'], max_length=64, truncation=True)['input_ids']
    return x
cot = lambda ds: Dataset.from_dict({'dau_vao': [x['dau_vao'] for x in ds], 'cau': [x['cau'] for x in ds]})
dtr = cot(tr).map(ma_hoa, batched=True, remove_columns=['dau_vao', 'cau'])
dva = cot(va).map(ma_hoa, batched=True, remove_columns=['dau_vao', 'cau'])

args = Seq2SeqTrainingArguments(
    output_dir='/kaggle/working/vit5-tam', per_device_train_batch_size=16, per_device_eval_batch_size=32,
    learning_rate=5e-4, num_train_epochs=4, warmup_ratio=0.05, weight_decay=0.01,
    eval_strategy='epoch', save_strategy='no', logging_steps=50, report_to='none',
    fp16=False)  # T5 de bi NaN voi fp16 tren T4
tr5 = Seq2SeqTrainer(model=m, args=args, train_dataset=dtr, eval_dataset=dva,
                     data_collator=DataCollatorForSeq2Seq(tk, model=m))
tr5.train()
tr5.save_model('/kaggle/working/vit5-ghep-cau'); tk.save_pretrained('/kaggle/working/vit5-ghep-cau')

In [ ]:
import chay_llm, gc
he = chay_llm.T5('/kaggle/working/vit5-ghep-cau')
for dv in ['nhieu', 'sach']:
    chay_llm.main(['--he', 't5:vit5-ghep-cau', '--dieu-kien', 'zero', '--dau-vao', dv], he=he)
del he, m, tr5; gc.collect(); torch.cuda.empty_cache()

## B. Qwen-2.5-Instruct + QLoRA (Unsloth)

Định dạng huấn luyện **giống hệt** lúc suy luận: system prompt điều kiện `zero` + đầu vào → JSON
`{"chon": [...], "cau": "..."}`. Sau khi train, mô hình tự học trật tự NNKH nên không cần quy tắc / ví dụ trong prompt.
Lưu ý: API của Unsloth / TRL thay đổi theo phiên bản — nếu ô dưới báo lỗi tham số, xem notebook mẫu Qwen2.5 mới nhất của Unsloth.

In [ ]:
!pip install -q unsloth

In [ ]:
from unsloth import FastLanguageModel
import json, torch
from datasets import Dataset
from prompt import tin_nhan, tra_loi_mau

MO_HINH_QWEN = 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit'   # thu ca '...-1.5B-...' va '...-3B-...'
model, tokenizer = FastLanguageModel.from_pretrained(MO_HINH_QWEN, max_seq_length=1024, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    use_gradient_checkpointing='unsloth', random_state=0)

def van_ban(x):
    ms = tin_nhan('zero', x['dau_vao'], []) + [{'role': 'assistant', 'content': tra_loi_mau(x)}]
    return tokenizer.apply_chat_template(ms, tokenize=False)
doc = lambda t: [json.loads(d) for d in open(t, encoding='utf-8')]
dtr = Dataset.from_dict({'text': [van_ban(x) for x in doc('du-lieu/sinh/train.jsonl')]})
dva = Dataset.from_dict({'text': [van_ban(x) for x in doc('du-lieu/sinh/val.jsonl')]})
print(dtr[0]['text'][-300:])

In [ ]:
from trl import SFTTrainer, SFTConfig
cau_hinh = SFTConfig(
    output_dir='/kaggle/working/qwen-tam', dataset_text_field='text', max_seq_length=1024,
    per_device_train_batch_size=4, gradient_accumulation_steps=2, num_train_epochs=1,
    learning_rate=2e-4, warmup_ratio=0.03, lr_scheduler_type='cosine', optim='adamw_8bit',
    fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported(),
    logging_steps=25, eval_strategy='steps', eval_steps=200, save_strategy='no', report_to='none', seed=0)
try:
    trainer = SFTTrainer(model=model, processing_class=tokenizer, train_dataset=dtr, eval_dataset=dva, args=cau_hinh)
except TypeError:  # TRL cu
    trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=dtr, eval_dataset=dva, args=cau_hinh)
trainer.train()
TEN = MO_HINH_QWEN.split('/')[-1]
model.save_pretrained(f'/kaggle/working/lora-{TEN}'); tokenizer.save_pretrained(f'/kaggle/working/lora-{TEN}')

In [ ]:
import chay_llm
FastLanguageModel.for_inference(model)

def he_qwen(ms):
    x = tokenizer.apply_chat_template(ms, add_generation_prompt=True, return_tensors='pt').to(model.device)
    with torch.no_grad():
        y = model.generate(input_ids=x, max_new_tokens=128, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(y[0, x.shape[1]:], skip_special_tokens=True)

for dv in ['nhieu', 'sach']:
    chay_llm.main(['--he', f'qlora:{TEN}', '--dieu-kien', 'zero', '--dau-vao', dv], he=he_qwen)

In [ ]:
from IPython.display import Markdown
!cd du-lieu && zip -qr /kaggle/working/ket-qua-muc2.zip ket-qua
Markdown(open('du-lieu/ket-qua/tong-ket.md', encoding='utf-8').read())